# 01 - Kiểm tra dữ liệu thật trên BigQuery

Chạy **trước khi** làm Dashboard để biết đúng tên bảng, tên cột và giá trị thật. Notebook chỉ **đọc** dữ liệu, không ghi gì lên BigQuery.

**Chuẩn bị (một lần):**
1. Chọn kernel là `.venv-1` (góc trên bên phải), cài thư viện: `pip install -r dashboard/requirements.txt ipykernel`
2. Đăng nhập Google bằng tài khoản đã được cấp quyền: `gcloud auth application-default login` (chạy trong terminal)

Chạy lần lượt từ trên xuống. Chạy xong, **lưu notebook (còn nguyên output)** và gửi cho trợ lý.

In [1]:
import sys, pathlib

# Tìm thư mục dashboard/ (chứa src/) để import được các module của app
here = pathlib.Path.cwd()
for base in [here, here.parent, here / "dashboard", here.parent / "dashboard"]:
    if (base / "src" / "config.py").exists():
        sys.path.insert(0, str(base))
        break
else:
    raise FileNotFoundError("Không tìm thấy dashboard/src. Hãy mở notebook từ bên trong repo.")
print("Dùng thư mục:", base)

Dùng thư mục: d:\olist-lakehouse-\ThanhTruc\dashboard_package\dashboard


In [2]:
import pandas as pd
from IPython.display import display
from google.cloud import bigquery
from src import config as cfg

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)
client = bigquery.Client(project=cfg.PROJECT_ID)
print("Kết nối thành công, project:", client.project)

d:\olist-lakehouse-\.venv-1\Lib\site-packages\google\auth\_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Kết nối thành công, project: olist-lakehouse-511407


## 1. Có những dataset và bảng nào?

In [3]:
for ds in client.list_datasets():
    print(ds.dataset_id)
    for t in client.list_tables(ds.dataset_id):
        print("    ", t.table_id)

olist_bronze
     category_translation
     customers
     geolocation
     order_items
     orders
     payments
     products
     reviews
     sellers
olist_gold
     dim_customers
     dim_products
     dim_sellers
     fct_order_items
     fct_orders
olist_ml
     ml_wide_table
     order_predictions
olist_silver
     stg_category_translation
     stg_customers
     stg_geolocation
     stg_order_items
     stg_orders
     stg_payments
     stg_products
     stg_reviews
     stg_sellers


## 2. Bảng dự đoán của TV3 (`order_predictions`)

In [4]:
pred = client.query(f"SELECT * FROM `{cfg.PRED_TABLE}`").to_dataframe(create_bqstorage_client=False)
print("Kích thước:", pred.shape)
display(pred.dtypes.to_frame("kiểu dữ liệu"))
display(pred.head())

Kích thước: (99441, 6)


,kiểu dữ liệu
order_id,str
late_probability,float64
predicted_late,Int64
risk_level,str
top_risk_factor_1,str
top_risk_factor_2,str


,order_id,late_probability,predicted_late,risk_level,top_risk_factor_1,top_risk_factor_2
0,0b8a71b3f5c221a69f4d23097cd604f6,0.1662,0,Low Risk,distance_km,customer_state_AL
1,cc97c71d908caa2c0b5754b35ebaccd8,0.1957,0,Low Risk,distance_km,customer_state_AL
2,4e506215592eafe8d7366eb7fa6cf532,0.2241,0,Low Risk,distance_km,customer_state_AL
3,1c3a0c1d7e9d92523cfe3bfad809a439,0.2548,0,Low Risk,purchase_month,customer_state_AL
4,ed18204b2bef82c1eeb3f13852c9dbdf,0.2859,0,Low Risk,distance_km,customer_state_AL


In [5]:
print("risk_level:")
display(pred["risk_level"].value_counts(dropna=False))
print("predicted_late:")
display(pred["predicted_late"].value_counts(dropna=False))
print("late_probability:")
display(pred["late_probability"].describe())

risk_level:


risk_level
Low Risk       56868
Medium Risk    31401
High Risk      11172
Name: count, dtype: int64

predicted_late:


predicted_late
0    77177
1    22264
Name: count, dtype: Int64

late_probability:


count    99441.000000
mean         0.356318
std          0.201577
min          0.010400
25%          0.201600
50%          0.314200
75%          0.474800
max          0.951600
Name: late_probability, dtype: float64

In [6]:
print("top_risk_factor_1 rỗng:", int(pred["top_risk_factor_1"].isna().sum()))
print("top_risk_factor_2 rỗng:", int(pred["top_risk_factor_2"].isna().sum()))
print("\ntop_risk_factor_1 (25 giá trị phổ biến nhất):")
display(pred["top_risk_factor_1"].value_counts().head(25))
print("top_risk_factor_2 (25 giá trị phổ biến nhất):")
display(pred["top_risk_factor_2"].value_counts().head(25))

top_risk_factor_1 rỗng: 0
top_risk_factor_2 rỗng: 0

top_risk_factor_1 (25 giá trị phổ biến nhất):


top_risk_factor_1
purchase_month                            27084
purchase_weekday                          10092
customer_state_SP                          9309
customer_state_RJ                          8068
distance_km                                7228
seller_prep_days                           5810
order_value                                3670
customer_state_MG                          3597
max_product_width_cm                       3457
seller_state_SP                            3244
total_product_weight_g                     2233
max_product_weight_g                       2067
seller_state_RJ                            2005
max_product_length_cm                      1499
customer_state_BA                          1481
customer_state_ES                          1338
max_product_height_cm                      1334
freight_value                              1122
product_category_name_beleza_saude          747
customer_state_SC                           613
item_count            

top_risk_factor_2 (25 giá trị phổ biến nhất):


top_risk_factor_2
customer_state_SP                         19509
seller_state_SP                           11484
customer_state_MG                         10608
purchase_weekday                           7937
seller_prep_days                           5741
max_product_width_cm                       5078
order_value                                4447
distance_km                                4271
total_product_weight_g                     3977
customer_state_RJ                          3619
item_count                                 2843
max_product_height_cm                      2528
max_product_weight_g                       2334
freight_value                              2215
purchase_month                             2064
max_product_length_cm                      1994
seller_state_RJ                            1263
customer_state_BA                          1084
product_category_name_beleza_saude         1005
product_category_name_moveis_decoracao      666
customer_state_SC     

## 3. Wide Table của TV2 (`ml_wide_table`)

In [7]:
wide = client.query(f"SELECT * FROM `{cfg.WIDE_TABLE}` LIMIT 5").to_dataframe(create_bqstorage_client=False)
display(wide.dtypes.to_frame("kiểu dữ liệu"))
display(wide.head(3))

,kiểu dữ liệu
order_id,str
item_count,Int64
order_value,object
freight_value,object
total_product_weight_g,float64
max_product_weight_g,float64
max_product_length_cm,float64
max_product_height_cm,float64
max_product_width_cm,float64
product_category_name,str


,order_id,item_count,order_value,freight_value,total_product_weight_g,max_product_weight_g,max_product_length_cm,max_product_height_cm,max_product_width_cm,product_category_name,customer_state,seller_state,purchase_month,purchase_weekday,seller_prep_days,distance_km,late_delivery
0,02bfe71c2bbd56f6de9c2c357adff45c,0,None,None,NaN,NaN,NaN,NaN,NaN,unknown,PR,unknown,12,0,NaN,NaN,<NA>
1,7d576f29bdf44b7857414baff1c788b1,0,None,None,NaN,NaN,NaN,NaN,NaN,unknown,PR,unknown,11,0,NaN,NaN,<NA>
2,91663de20f19686d62e4dbc0087fcf62,0,None,None,NaN,NaN,NaN,NaN,NaN,unknown,RJ,unknown,7,6,NaN,NaN,<NA>


## 4. Ghép hai bảng theo `order_id` có khớp không?

In [8]:
join_check = client.query(f"""
SELECT COUNT(*) AS so_dong_sau_join, COUNT(DISTINCT p.order_id) AS so_order_khac_nhau
FROM `{cfg.PRED_TABLE}` p JOIN `{cfg.WIDE_TABLE}` w USING (order_id)
""").to_dataframe(create_bqstorage_client=False)
display(join_check)

,so_dong_sau_join,so_order_khac_nhau
0,99441,99441


## 5. Nhãn thật so với mức rủi ro
Cho biết trong nhóm *High Risk*, bao nhiêu % đơn thật sự trễ.

In [9]:
cmp_ = client.query(f"""
SELECT w.late_delivery, p.risk_level, COUNT(*) AS n, ROUND(AVG(p.late_probability), 3) AS avg_p
FROM `{cfg.PRED_TABLE}` p JOIN `{cfg.WIDE_TABLE}` w USING (order_id)
GROUP BY 1, 2 ORDER BY 1, 2
""").to_dataframe(create_bqstorage_client=False)
display(cmp_)

hi = cmp_[(cmp_["risk_level"] == cfg.RISK_HIGH) & cmp_["late_delivery"].notna()]
if not hi.empty:
    late = hi.loc[hi["late_delivery"] == 1, "n"].sum()
    print(f"Trong nhóm High Risk có nhãn: {late}/{hi['n'].sum()} đơn thật sự trễ ({late / hi['n'].sum():.1%})")

,late_delivery,risk_level,n,avg_p
0,<NA>,High Risk,374,0.767
1,<NA>,Low Risk,1616,0.222
2,<NA>,Medium Risk,975,0.472
3,0,High Risk,7224,0.748
4,0,Low Risk,54711,0.213
5,0,Medium Risk,28006,0.466
6,1,High Risk,3574,0.786
7,1,Low Risk,541,0.267
8,1,Medium Risk,2420,0.517


Trong nhóm High Risk có nhãn: 3574/10798 đơn thật sự trễ (33.1%)


## 6. Tỷ lệ trễ thật theo năm-tháng
Dùng để kiểm tra yếu tố `purchase_month` (bảng `olist_gold.fct_orders`).

In [10]:
try:
    ym = client.query(f"""
    SELECT FORMAT_DATE('%Y-%m', DATE(order_purchase_timestamp)) AS nam_thang,
           COUNT(*) AS so_don,
           ROUND(SAFE_DIVIDE(COUNTIF(late_delivery = 1), COUNTIF(late_delivery IS NOT NULL)), 4) AS ty_le_tre
    FROM `{cfg.ORDERS_TABLE}` GROUP BY 1 ORDER BY 1
    """).to_dataframe(create_bqstorage_client=False)
    display(ym)
except Exception as e:
    print("Không đọc được", cfg.ORDERS_TABLE, "->", type(e).__name__, str(e)[:300])

,nam_thang,so_don,ty_le_tre
0,2016-09,4,1.0000
1,2016-10,324,0.0074
2,2016-12,1,0.0000
3,2017-01,800,0.0293
4,2017-02,1780,0.0296
5,2017-03,2682,0.0456
6,2017-04,2404,0.0656
7,2017-05,3700,0.0299
8,2017-06,3245,0.0303
9,2017-07,4026,0.0279


**Xong.** Lưu notebook (Ctrl+S) rồi gửi file `.ipynb` này (còn output) cho trợ lý.